# Top 6 Model Runtime Prep

이 노트북은 FastAPI용 모델 파일을 바로 생성하지 않고, 상위 6개 스테이션 기준으로 입력 경로와 산출물 구조를 점검하기 위한 준비용 노트북입니다.

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work/hmw3')
NOTE_DIR = BASE_DIR / 'Note'
DATA_DIR = BASE_DIR / 'Data'
SCRIPT_PATH = BASE_DIR / 'scripts' / 'generate_top20_station_suite.py'

TOP6_STATION_IDS = [2348, 2335, 2377, 2384, 2306, 2375]

paths = {
    'top5_trend_notebook': NOTE_DIR / 'hmw_top5_station_trends_2023_2025.ipynb',
    'sample_model_notebook': NOTE_DIR / 'hmw2332.ipynb',
    'top20_metric_summary': DATA_DIR / 'top20_station_metrics_summary.csv',
    'suite_script': SCRIPT_PATH,
}

pd.DataFrame([
    {'name': name, 'path': str(path), 'exists': path.exists()}
    for name, path in paths.items()
])

In [ ]:
station_file_rows = []

for station_id in TOP6_STATION_IDS:
    station_file_rows.append({
        'station_id': station_id,
        'raw_csv': str(DATA_DIR / f'station_{station_id}.csv'),
        'raw_csv_exists': (DATA_DIR / f'station_{station_id}.csv').exists(),
        'metric_csv': str(DATA_DIR / f'station_{station_id}_offday_month_ridge_metrics.csv'),
        'metric_csv_exists': (DATA_DIR / f'station_{station_id}_offday_month_ridge_metrics.csv').exists(),
        'formula_csv': str(DATA_DIR / f'station_{station_id}_offday_hour_formulas.csv'),
        'formula_csv_exists': (DATA_DIR / f'station_{station_id}_offday_hour_formulas.csv').exists(),
        'weight_csv': str(DATA_DIR / f'station_{station_id}_month_weights.csv'),
        'weight_csv_exists': (DATA_DIR / f'station_{station_id}_month_weights.csv').exists(),
    })

station_file_df = pd.DataFrame(station_file_rows)
station_file_df

In [ ]:
metric_path = DATA_DIR / 'top20_station_metrics_summary.csv'
metric_df = pd.read_csv(metric_path)

test_df = metric_df[
    (metric_df['split'] == 'test')
    & (metric_df['target'].isin(['rental_count', 'return_count']))
].copy()

r2_df = test_df.pivot_table(index='station_id', columns='target', values='r2', aggfunc='mean').reset_index()
rmse_df = test_df.pivot_table(index='station_id', columns='target', values='rmse', aggfunc='mean').reset_index().rename(columns={'rental_count': 'rental_rmse', 'return_count': 'return_rmse'})
mae_df = test_df.pivot_table(index='station_id', columns='target', values='mae', aggfunc='mean').reset_index().rename(columns={'rental_count': 'rental_mae', 'return_count': 'return_mae'})

ranking_df = r2_df.merge(rmse_df, on='station_id').merge(mae_df, on='station_id')
ranking_df['combined_test_r2'] = ranking_df[['rental_count', 'return_count']].mean(axis=1)
ranking_df['combined_test_rmse'] = ranking_df[['rental_rmse', 'return_rmse']].mean(axis=1)
ranking_df['combined_test_mae'] = ranking_df[['rental_mae', 'return_mae']].mean(axis=1)
ranking_df = ranking_df.sort_values('combined_test_r2', ascending=False).reset_index(drop=True)

ranking_df.head(6)

In [ ]:
required_suffixes = [
    '.csv',
    '_offday_month_ridge_metrics.csv',
    '_offday_hour_formulas.csv',
    '_month_weights.csv',
    '_offday_month_ridge_coefficients.csv',
    '_feature_importance.csv',
]

component_rows = []
for station_id in TOP6_STATION_IDS:
    for suffix in required_suffixes:
        filename = f'station_{station_id}{suffix}'
        path = DATA_DIR / filename
        component_rows.append({
            'station_id': station_id,
            'component_file': filename,
            'exists': path.exists(),
        })

component_df = pd.DataFrame(component_rows)
component_df

In [ ]:
bundle_preview_df = pd.DataFrame([
    {
        'station_id': station_id,
        'bundle_filename': f'station_{station_id}_runtime_bundle.joblib',
        'targets': 'rental_count, return_count',
        'contains_formula': True,
        'contains_weights': True,
        'contains_coefficients': True,
        'contains_feature_schema': True,
        'recommended_output_dir': '/Users/cheng80/Desktop/ddri_web/fastapi/app/models/runtime',
    }
    for station_id in TOP6_STATION_IDS
])

bundle_preview_df

## 다음 단계 메모

- 현재는 점검만 수행합니다.
- 상위 6개 기준과 필수 파일 매트릭스는 이 노트북에서 확인할 수 있습니다.
- 다음 단계는 스테이션당 1개 번들 `joblib` 스키마를 확정하는 것입니다.
- 별도 재실행 스크립트는 `top6_model_runtime_export.py`를 사용합니다.